# DantinoX — Colab Quickstart

> *"E quindi uscimmo a riveder le stelle."*

This notebook trains a decoder-only Transformer with **DantinoX** on a HuggingFace dataset, generates text, and optionally pushes the checkpoint to the Hub.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or A100 if available).

| Step | What happens |
|---|---|
| **1 — Install** | JAX CUDA + DantinoX from PyPI |
| **2 — Configure** | Small model that fits comfortably on a T4 |
| **3 — Train** | ~5 min on T4, streams dataset from HuggingFace Hub |
| **4 — Generate** | Single prompt, batch, and streaming |
| **5 — Hub** | Push/pull checkpoint *(optional)* |
| **6 — LoRA** | Fine-tune with adapters *(optional)* |

---
## 1 — Install

We upgrade JAX to the CUDA 12 build (Colab ships an older CPU-only version) and install DantinoX from PyPI.

> ⚠️ **Restart the runtime once after this cell** (`Runtime → Restart session`), then continue from cell 2.

In [ ]:
# Upgrade JAX to CUDA 12 build — must restart runtime after this
!pip install -q -U "jax[cuda12]" jaxlib

# Install DantinoX + optional extras from GitHub
!pip install -q "dantinox[data,hub]"

print("\n✅ Done — restart the runtime now (Runtime → Restart session), then run from the next cell.")

---
## 2 — Check GPU

In [ ]:
import jax

devices = jax.devices()
print("JAX devices:", devices)

if not any("cuda" in str(d).lower() or "gpu" in str(d).lower() for d in devices):
    print("\n⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"\n✅ Running on {devices[0]}")

---
## 3 — Configure

A model that trains in ~5 minutes on a T4 and produces coherent Italian text.
Tweak `num_blocks` / `dim` for larger/smaller experiments.

In [ ]:
from dantinox import Config

config = Config(
    # ── Architecture ─────────────────────────────────────────────
    dim=256,
    n_heads=8,
    head_size=32,        # dim == n_heads * head_size
    num_blocks=6,
    max_context=256,
    kv_heads=2,          # GQA: 4 query heads per KV head
    use_swiglu=True,
    norm_type="rmsnorm",
    use_flash_attention=True,

    # ── Dataset — streamed from HuggingFace ──────────────────────
    dataset_source="huggingface",
    dataset_name="Daniele/dante-corpus",
    dataset_text_field="text",
    streaming=True,

    # ── Training ─────────────────────────────────────────────────
    epochs=300,
    batch_size=64,
    grad_accum=4,         # effective batch = 256 tokens * 64 = 16 384
    lr=3e-4,
    lr_schedule="wsd",    # warmup → stable → cosine decay
    warmup_steps=100,
    grad_clip=1.0,
    patience=10,          # early stopping
    use_bf16=True,        # faster + less VRAM on Ampere / T4
)

print(config)

---
## 4 — Train

In [ ]:
from dantinox import Trainer

trainer = Trainer(config)
run_dir = trainer.fit()   # no data_path — streams from HuggingFace

print(f"\nCheckpoint saved to: {run_dir}")

---
## 5 — Generate

In [ ]:
from dantinox import Generator

gen = Generator(run_dir, seed=42)

# ── Single prompt ─────────────────────────────────────────────────
text = gen.generate(
    "Nel mezzo del cammin ",
    max_new_tokens=200,
    temperature=0.8,
    top_k=40,
)
print(text)

In [ ]:
# ── Batch generation — one forward pass ──────────────────────────
prompts = [
    "Nel mezzo del cammin ",
    "Lasciate ogni speranza ",
    "Per me si va nella città dolente ",
]

outputs = gen.generate_batch(prompts, max_new_tokens=80, temperature=0.9)
for p, o in zip(prompts, outputs):
    print(f"▶ {p!r}")
    print(o)
    print()

In [ ]:
# ── Streaming — tokens printed as they arrive ─────────────────────
print("Streaming: ", end="", flush=True)
for chunk in gen.stream("Nel mezzo del cammin ", max_new_tokens=150, temperature=0.8):
    print(chunk, end="", flush=True)
print()

---
## 6 — HuggingFace Hub *(optional)*

Push your checkpoint so anyone can load it with `Generator("your-name/your-model")`.

You need a [HuggingFace account](https://huggingface.co/join) and a write token.

In [ ]:
HF_TOKEN  = "hf_..."          # paste your HF token here
HF_REPO   = "your-name/dantinox-dante"   # repo to create / update

from dantinox import push
url = push(run_dir, HF_REPO, token=HF_TOKEN, private=False)
print(f"Pushed to: {url}")

In [ ]:
# Load directly from the Hub on any machine — no pull step needed
gen_hub = Generator(HF_REPO, token=HF_TOKEN)
print(gen_hub.generate("Nel mezzo del cammin ", max_new_tokens=100))

---
## 7 — LoRA Fine-Tuning *(optional)*

Fine-tune the checkpoint on a different corpus. Only ~0.2 % of parameters are trained — base weights stay completely frozen.

In [ ]:
import os
from dantinox import Config, Trainer, Generator

# Build a fine-tuning config from the saved config
ft_config = Config.from_yaml(os.path.join(run_dir, "config.yaml"))
ft_config.use_lora      = True
ft_config.lora_rank     = 8
ft_config.lora_alpha    = 16.0
ft_config.lora_targets  = "attention"   # "attention" | "mlp" | "all"
ft_config.epochs        = 50
ft_config.lr            = 1e-4          # lower LR for fine-tuning

# Fine-tune on a different dataset
ft_config.dataset_name  = "Daniele/dante-corpus"   # swap to your corpus

ft_run = Trainer(ft_config).fit()

# Generate from the fine-tuned model
ft_gen = Generator(ft_run, seed=42)
print(ft_gen.generate("Nel mezzo del cammin ", max_new_tokens=200, temperature=0.8))

---
## Tips

### Bigger model for A100

```python
config = Config(
    dim=512, n_heads=16, head_size=32, num_blocks=12,
    max_context=512, kv_heads=4,
    use_bf16=True, use_flash_attention=True,
    norm_type="rmsnorm", lr_schedule="wsd",
    dataset_source="huggingface",
    dataset_name="Daniele/dante-corpus",
    streaming=True,
    epochs=1000, batch_size=128, grad_accum=8, lr=3e-4, grad_clip=1.0,
)
```

### Different datasets

| Dataset | `dataset_name` | `dataset_config` | `dataset_text_field` |
|---|---|---|---|
| WikiText-103 | `wikitext` | `wikitext-103-v1` | `text` |
| TinyStories | `roneneldan/TinyStories` | *(empty)* | `text` |
| OpenWebText | `openwebtext` | *(empty)* | `text` |
| C4 (English) | `allenai/c4` | `en` | `text` |

### Resume an interrupted run

```python
run_dir = Trainer(config).fit(resume=True, run_dir="runs/run_20260101_120000")
```

### LR finder (before committing to a long run)

```python
lr, lrs, losses = Trainer(config).find_lr(num_steps=100, plot=False)
print(f"Suggested LR: {lr:.2e}")
```